# Workshop — 4. Evaluate the Fine-Tuned SmolVLA

Notebook 2 established the zero-shot baseline. Notebook 3 produced an updated checkpoint on SageMaker. This notebook downloads that artifact and evaluates both checkpoints under the same conditions:

- the same SO100 MuJoCo scene;
- the same `top` and `wrist` cameras;
- the same language instruction;
- 400 control steps at 30 Hz;
- the same `placed_in_box` success metric.

The two policies are loaded sequentially so the laptop does not keep two 450M-parameter models in memory.

## 1. Install the Evaluation Runtime

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os
import platform

os.environ.setdefault("MUJOCO_GL", "cgl" if platform.system() == "Darwin" else "egl")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("STRANDS_TRUST_REMOTE_CODE", "1")
if platform.system() == "Darwin":
    os.environ.setdefault("DYLD_FALLBACK_LIBRARY_PATH", "/opt/homebrew/lib")

## 2. Download the SageMaker Artifact

The SageMaker job used uncompressed output, so the checkpoint files are directly addressable under its model artifact prefix. The cells below find the latest completed job by base-job prefix, read `ModelArtifacts.S3ModelArtifacts` from SageMaker, and append only the exported `smolvla/` directory. The downloader ignores SageMaker's zero-byte upload marker objects.

In [ ]:
from pathlib import Path

import boto3

TRAINING_JOB_PREFIX = "train-smolvla-so100-sim"
TRAINING_JOB_NAME = None  # Set an exact completed job name to override automatic discovery.


def get_last_job_name(job_name_prefix):
    sagemaker_client = boto3.client("sagemaker")
    matching_jobs = []
    next_token = None

    while True:
        search_params = {
            "Resource": "TrainingJob",
            "SearchExpression": {
                "Filters": [
                    {
                        "Name": "TrainingJobName",
                        "Operator": "Contains",
                        "Value": job_name_prefix,
                    },
                    {
                        "Name": "TrainingJobStatus",
                        "Operator": "Equals",
                        "Value": "Completed",
                    },
                ]
            },
            "SortBy": "CreationTime",
            "SortOrder": "Descending",
            "MaxResults": 100,
        }
        if next_token:
            search_params["NextToken"] = next_token

        response = sagemaker_client.search(**search_params)
        matching_jobs.extend(
            result["TrainingJob"]["TrainingJobName"]
            for result in response["Results"]
            if result["TrainingJob"]["TrainingJobName"].startswith(job_name_prefix)
        )
        next_token = response.get("NextToken")
        if matching_jobs or not next_token:
            break

    if not matching_jobs:
        raise ValueError(
            f"No completed training jobs found with prefix {job_name_prefix!r}"
        )
    return matching_jobs[0]


sagemaker_client = boto3.client("sagemaker")
training_job_name = TRAINING_JOB_NAME or get_last_job_name(TRAINING_JOB_PREFIX)
job_description = sagemaker_client.describe_training_job(
    TrainingJobName=training_job_name
)
if job_description["TrainingJobStatus"] != "Completed":
    raise RuntimeError(
        f"Training job {training_job_name} is {job_description['TrainingJobStatus']}, not Completed"
    )

model_artifact_root = job_description["ModelArtifacts"]["S3ModelArtifacts"].rstrip("/")
S3_MODEL_URI = f"{model_artifact_root}/smolvla/"
LOCAL_MODEL_DIR = Path(
    "outputs/smolvla-so100/evaluations"
) / training_job_name / "model"
LOCAL_MODEL_DIR = LOCAL_MODEL_DIR.resolve()

print("Training job:", training_job_name)
print("Remote model:", S3_MODEL_URI)
print("Local model: ", LOCAL_MODEL_DIR)

In [ ]:
from urllib.parse import urlparse


def download_s3_prefix(s3_uri, destination):
    parsed = urlparse(s3_uri)
    if parsed.scheme != "s3" or not parsed.netloc:
        raise ValueError(f"Expected an s3:// URI, got {s3_uri!r}")

    bucket = parsed.netloc
    prefix = parsed.path.lstrip("/")
    if prefix and not prefix.endswith("/"):
        prefix += "/"

    client = boto3.client("s3")
    paginator = client.get_paginator("list_objects_v2")
    downloaded = []
    destination.mkdir(parents=True, exist_ok=True)

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for item in page.get("Contents", []):
            key = item["Key"]
            relative = key[len(prefix):]
            if not relative or relative.endswith("/"):
                continue
            if relative.endswith((".sagemaker-uploaded", ".sagemaker-uploading")):
                continue

            local_path = destination / relative
            local_path.parent.mkdir(parents=True, exist_ok=True)
            client.download_file(bucket, key, str(local_path))
            downloaded.append((relative, int(item["Size"])))

    if not downloaded:
        raise FileNotFoundError(f"No model files found under {s3_uri}")
    return downloaded


downloaded = download_s3_prefix(S3_MODEL_URI, LOCAL_MODEL_DIR)
print(f"Downloaded {len(downloaded)} files ({sum(size for _, size in downloaded) / 2**20:.1f} MiB)")
for name, size in downloaded:
    print(f"  {size:>12,}  {name}")

## 3. Validate the Checkpoint Contract

The fine-tuned checkpoint must consume the simulation-native camera keys and use the statistics exported by the training job. Notebook 2's degrees/0–100 embodiment conversion must not be reused here: the simulation dataset was recorded in radians.

In [ ]:
import json

required_files = {
    "config.json",
    "model.safetensors",
    "policy_preprocessor.json",
    "policy_preprocessor_step_5_normalizer_processor.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor_step_0_unnormalizer_processor.safetensors",
    "train_config.json",
    "training_manifest.json",
}
missing = sorted(name for name in required_files if not (LOCAL_MODEL_DIR / name).is_file())
if missing:
    raise FileNotFoundError(f"Incomplete model artifact: {missing}")

config = json.loads((LOCAL_MODEL_DIR / "config.json").read_text())
manifest = json.loads((LOCAL_MODEL_DIR / "training_manifest.json").read_text())
input_features = config["input_features"]

assert config["type"] == "smolvla"
assert input_features["observation.state"]["shape"] == [6]
assert config["output_features"]["action"]["shape"] == [6]
assert {
    key for key, feature in input_features.items() if feature["type"] == "VISUAL"
} == {"observation.images.top", "observation.images.wrist"}
assert manifest["state_action_units"] == "radians"

print(json.dumps({
    "base_model": manifest["base_model"],
    "base_revision": manifest["base_revision"],
    "dataset_episodes": manifest["dataset_episodes"],
    "dataset_frames": manifest["dataset_frames"],
    "camera_keys": manifest["camera_keys"],
    "state_action_units": manifest["state_action_units"],
    "model_size_mib": round((LOCAL_MODEL_DIR / "model.safetensors").stat().st_size / 2**20, 1),
}, indent=2))

## 4. Common Evaluation Setup

Both policies use the scene builder and physical success metric from Notebook 2.

In [ ]:
import gc
import sys

import torch
from IPython.display import HTML, Video, display

sys.path.insert(0, str(Path("code").resolve()))

from strands_robots.policies import create_policy
from vla_pick import (
    INSTRUCTION,
    JOINT_KEYS,
    MODEL_ID,
    MODEL_REVISION,
    build_scene,
    choose_device,
    cube_diagnostics,
    load_policy,
    run_rollout,
)

device = choose_device("auto")
BASELINE_VIDEO = Path("smolvla_baseline_eval.mp4").resolve()
FINETUNED_VIDEO = Path("smolvla_finetuned_eval.mp4").resolve()

print("Device:", device)
print("Instruction:", INSTRUCTION)

## 5. Reproduce the Zero-Shot Baseline

This step recreates the experiment from Notebook 2 before evaluating the adapted model:

1. `build_scene()` creates a fresh SO100 simulation with the same cube, target box, cameras, and initial pose used throughout the workshop.
2. `load_policy()` loads the original checkpoint trained on real SO100 demonstrations.
3. The original embodiment adapter maps the simulation cameras to the checkpoint's `camera1`/`camera2` inputs and converts MuJoCo radians to the real dataset's degree and 0–100 gripper representation.
4. `run_rollout()` executes 400 control steps at 30 Hz and records `smolvla_baseline_eval.mp4`.
5. `cube_diagnostics()` records the final cube position and computes `placed_in_box`.

`run_policy status="success"` only means that inference and action execution completed without a software error. The physical baseline succeeds only when `placed_in_box=True`.

After collecting the result, the following cell deletes the baseline policy and simulation and clears the accelerator cache. This prevents both 450M-parameter models from occupying memory at the same time.

In [ ]:
baseline_sim = build_scene()
baseline_policy, _ = load_policy(device)
baseline_result = run_rollout(
    baseline_sim,
    baseline_policy,
    steps=400,
    video_path=BASELINE_VIDEO,
)
baseline_diagnostics = cube_diagnostics(baseline_sim)

print("checkpoint:", f"{MODEL_ID}@{MODEL_REVISION[:8]}")
print("run_policy status:", baseline_result["status"])
print("task diagnostics:", baseline_diagnostics)

In [ ]:
del baseline_policy
del baseline_sim
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Baseline model released from memory.")

## 6. Evaluate the Fine-Tuned Checkpoint

This step repeats the same physical experiment with the checkpoint downloaded from the completed SageMaker Training Job:

1. A new embodiment adapter maps `wrist` and `top` directly to `observation.images.wrist` and `observation.images.top`.
2. State and action units remain in raw MuJoCo radians because the fine-tuning dataset and the exported normalization statistics use radians. Reusing Notebook 2's degree/0–100 conversion would normalize the new checkpoint incorrectly.
3. `create_policy()` loads the local fine-tuned model together with its new preprocessor, postprocessor, and simulation statistics.
4. `build_scene()` creates another fresh scene rather than continuing from the baseline's final robot or cube state.
5. The policy receives the same instruction and runs for the same 400 steps at 30 Hz.
6. The rollout is recorded as `smolvla_finetuned_eval.mp4`, and `cube_diagnostics()` computes the same `placed_in_box` metric.

Because robot, scene, instruction, cameras, rollout length, and success metric are unchanged, the checkpoint is the experimental variable being compared.

In [ ]:
finetuned_embodiment = {
    "name": "so100_smolvla_sim_finetuned",
    "obs_rename": {
        "wrist": "observation.images.wrist",
        "top": "observation.images.top",
    },
    "state_keys": JOINT_KEYS,
    "action_keys": JOINT_KEYS,
    "dim_policy": "strict",
    "state_units": "radians",
    "action_units": "radians",
}

finetuned_policy = create_policy(
    "lerobot_local",
    pretrained_name_or_path=str(LOCAL_MODEL_DIR),
    policy_type="smolvla",
    device=device,
    embodiment=finetuned_embodiment,
    strict_keys=True,
)
finetuned_sim = build_scene()
finetuned_result = run_rollout(
    finetuned_sim,
    finetuned_policy,
    steps=400,
    video_path=FINETUNED_VIDEO,
)
finetuned_diagnostics = cube_diagnostics(finetuned_sim)

print("checkpoint:", LOCAL_MODEL_DIR)
print("run_policy status:", finetuned_result["status"])
print("task diagnostics:", finetuned_diagnostics)

## 7. Before/After Comparison

`run_policy status` measures software execution. `placed_in_box` measures task success.

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "checkpoint": "zero-shot real-data checkpoint",
        "run_status": baseline_result["status"],
        "cube_position_m": baseline_diagnostics["cube_position_m"],
        "placed_in_box": bool(baseline_diagnostics["placed_in_box"]),
        "video": str(BASELINE_VIDEO),
    },
    {
        "checkpoint": "simulation fine-tuned checkpoint",
        "run_status": finetuned_result["status"],
        "cube_position_m": finetuned_diagnostics["cube_position_m"],
        "placed_in_box": bool(finetuned_diagnostics["placed_in_box"]),
        "video": str(FINETUNED_VIDEO),
    },
])
display(comparison)

display(HTML("<h3>Zero-shot baseline</h3>"))
display(Video(str(BASELINE_VIDEO), embed=True, width=640))
display(HTML("<h3>Simulation fine-tuned</h3>"))
display(Video(str(FINETUNED_VIDEO), embed=True, width=640))

## Interpretation

A successful fine-tuned rollout closes the workshop loop: matching demonstrations can correct the visual and embodiment distribution shift observed in Notebook 2.

A failed rollout does not invalidate the training job. Eight episodes are sufficient to validate the engineering pipeline but remain a very small learning dataset. Inspect the video and cube trajectory, then expand the successful demonstration set before changing model architecture.